# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Does a page's search position and visibility reliably predict whether its click-through rate will underperform — and can a simple learned model out-prioritize a hand-written rule at flagging the pages worth a human's review time?

**Decision this supports:** Which ranking signals a content/SEO team should actually trust when deciding where to spend limited manual review time, instead of defaulting to untested assumptions like "high volume = high priority" or "older content = declining." This is the FlyRank content problem behind my lane, Ranking Signal Analysis — a content team can't manually review every page every month, so before building any prioritization tool, someone has to establish which signals are trustworthy enough to prioritize on.

**Action:** A strategist reviews the resulting ranked queue and chooses between a CTR-focused fix or a content-quality review per flagged page.

**Cost of a wrong call:** Flagging a weak signal as strong sends review effort toward pages that don't need it, while genuinely underperforming pages go unaddressed.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank Internship Warehouse, Hugging Face (`FlyRank/internship-warehouse`).

**Table:** `fact_content_daily_performance` — grain of one row per report date, per pseudonymized client, per pseudonymized content item. 78,835,655 rows at full scale.

**Date window:** `month = '2026-03'` — a deliberately chosen mid-panel month, filtered to 3,611,061 rows. The `_sample` partition (June 2026, the sealed final month) was avoided entirely for feature/label development, since it's the natural outcome window for any past-predicts-future setup — used only for testing query mechanics.

**Excluded, and why:** every `ga4_*`, `sessions_*`, `ai_*`, and `scroll_events` column — all showed 0 non-null values in this slice, meaning no GA4, session-source, or AI-referral data was available for this period. `gsc_clicks` was excluded as a *feature* (though retained for label derivation) since it directly composes the CTR target — including it as an input feature would be label leakage.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** search position and impression volume are knowable at decision time and are not derived from the outcome being predicted.

**Features:** `gsc_avg_position`, `gsc_impressions`.

**Label definition:** `y_low_ctr` = 1 if a row's CTR (`clicks / impressions`) falls below the overall average CTR computed across the March slice, else 0. Derived, not provided — no ready-made trend/label column exists in the warehouse table.

**Baseline:** hand-written rule — `gsc_avg_position >= 21 AND gsc_impressions >= 50` — scored as a classifier against the same held-out test set as the model.

**Validation design:** grouped split by `content_hash_id` (scikit-learn `GroupShuffleSplit`, 80/20), chosen because the data's page × day grain means the same pages repeat across many dates (8 of 20 rows in an early review were a single page on different days). A random row split would let a page appear in both train and test, letting a model memorize rather than generalize. Verified — not assumed — zero page overlap between the resulting sets.

**Leakage checks:** `gsc_clicks` excluded from features (numerator of the CTR that defines the target). Confirmed no column in the source schema matches `trend`/`label`/`target` naming. Disclosed one partial structural overlap: `gsc_impressions` is both a feature and the denominator of the CTR the target is built from — not full leakage, but not fully independent either.

**Model:** Logistic Regression, `class_weight='balanced'` to correct for target imbalance (~90% of rows label low-CTR). Chosen as a proportionate, interpretable step up from the rule rather than an unjustified jump in complexity.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same test split, same metrics, both scored against the identical grouped holdout:

| Approach | Precision | Recall | F1 |
|---|---|---|---|
| Baseline (hand-written rule) | 0.858 | 0.048 | 0.092 |
| Logistic Regression | 0.957 | 0.847 | 0.898 |

The gap is almost entirely recall: the rule's strict threshold caught under 5% of genuinely low-CTR pages; the model caught 85%, without sacrificing precision.

**Confusion matrix (Logistic Regression):** TN 46,642 · FP 24,695 · FN 100,024 · TP 551,670. Even at 0.847 recall, 100,024 low-CTR pages were still missed — a strong F1 doesn't mean zero misses.

**Supporting signal findings:** CTR by position is flat from 1–20 (0.38% / 0.32% / 0.31%) and collapses at 21+ (0.13%) — a cliff, not a gradual slope. High-impression pages (avg. position 11.89) ranked no better than medium-impression pages (11.02); low-impression pages ranked clearly worse (16.85) — search volume alone was a weak predictor of ranking quality.

## 5. Limitations

*What this work cannot claim.*

- **Single month** (March 2026) — no seasonal or trend behavior captured.
- **No causal claim** — every finding is an observed, directional association in this sample, not proof that acting on a flag recovers clicks.
- **Partial feature/target overlap** — `gsc_impressions` sits on both sides of the model; scores should be read as directional priority, not an independent probability.
- **GSC-only signal surface** — GA4, session, and AI-referral data were entirely empty; nothing here speaks to engagement, conversion, or AI traffic.
- **No revenue weighting** — the queue ranks by impressions, not business value.
- **A correction we made on ourselves:** an early 100-row sample suggested this analysis was effectively single-client. The full Week 7 action queue surfaces at least five distinct clients in its top 20 rows alone — the corrected picture is stated here rather than the earlier assumption.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Two-reason-code action queue, exported to `work/outputs/action_playbook_queue.csv` (708,842 rows):

1. **POOR_POSITION_WITH_VISIBILITY → improve_ctr** — position 21+, CTR below the March average, impressions ≥ 50.
2. **ZERO_CLICKS_WITH_VISIBILITY → review_content_quality** — real impressions, zero clicks; may signal a relevance/quality problem a snippet fix won't solve.

**Human review required:** confirm the flag isn't a one-day anomaly; check query intent before recommending a CTR fix; rule out pages that should be consolidated/noindexed rather than fixed; verify the page isn't already performing well relative to peers.

**Never automated:** auto-publishing content changes, auto-removing/noindexing pages, auto-reallocating budget from rank order, treating rank as guaranteed ROI.

**Monitoring triggers:** the position-21 cliff shifts in a new month; the portfolio CTR benchmark moves; acted-on pages show no lift after ~30 days; a growing share of flags turn out already-healthy on review. Recommended cadence: monthly re-run, human-reviewed — not an automated retrain pipeline.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

- `figures/ctr_by_position_bucket.png` — CTR-by-position bar chart (embedded in the paper's Results section)
- `work/outputs/metrics.json` — model vs. baseline table, confusion matrix, coefficients, reason-code counts (the receipts every number in the paper traces back to)
- `work/outputs/action_playbook_queue.csv` — the full ranked action queue (regenerated on each run, not committed per the CI leak-guard)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
